# Changepoint features: what `add_changepoint_features` does

A raw changepoint mask is 0/1 — "here is a trough" — and on its own that is a poor model
input: only a handful of frames are ever 1. `ethograph.features.changepoints.add_changepoint_features`
expands one mask into the richer representation `more_changepoint_features` was built for:

- a multi-scale **Laplacian proximity** curve, one per sigma — "how close is the nearest changepoint";
  optionally scaled by a signal such as speed (`scale_by`) to emphasise changepoints where it is low
- two **offset** columns — samples *since* the previous and *until* the next changepoint, clipped at a
  horizon — so a frame knows which side of its nearest candidate it is on, which a symmetric kernel cannot say
- a **length** column — the log length of the candidate segment the frame sits in, the long range the
  offsets saturate on

This notebook is pure visualisation: it runs the function on one trial of the public
crow dataset {cite:p}`moll2025crows` so you can see
what each output column looks like before deciding whether to use it. Two ways to turn it
on for real (see `CLAUDE.md`'s segmentation-pipeline section):

1. **On your own `TrialTree`** — call `add_changepoint_features` yourself, same as any
   other feature-engineering step (`add_changepoints_to_ds`, `features/geometry.py`), then save.
   See the last cell.
2. **In `project.yaml`** — set `features.changepoint_features` (sigmas + distribution) and
   the segmentation pipeline expands every changepoint mask once per session, at
   `materialise`/`infer` time, without touching the source file.

## Download example data

In [1]:
from pathlib import Path

from ethograph.utils.download import download_example_dataset

try:
    _here = Path(__vsc_ipynb_file__).parent  # VS Code
except NameError:
    _here = Path().resolve()  # Jupyter Lab / Notebook (CWD = notebook dir)

data_folder = _here.parent / "data" / "Moll2025"
download_example_dataset("moll2025", data_folder)

print(f"\ndata_folder: {data_folder}")

  2024-12-18_041_Crow1-cam-1_s3d.npy (11/11)
  mapping: c:\Users\aksel\Documents\Code\ethograph\data\Moll2025\.ethograph\mapping.txt

data_folder: c:\Users\aksel\Documents\Code\ethograph\data\Moll2025


## Load one trial's speed + its changepoint mask

In [ ]:
import matplotlib.pyplot as plt

import ethograph as eto
from ethograph.features.changepoints import add_changepoint_features
from ethograph.io import schema

TRIAL = None          # None -> first trial
KEYPOINT = "beakTip"
SIGMAS = [0.56, 1.13, 2.25]  # kernel widths, in samples

dt = eto.open(str(data_folder / "Trial_data.nc"))
trial = TRIAL if TRIAL is not None else dt.trials[0]
ds = dt.trial(trial)

individual = ds.individual.values[0]
pinned = ds[["speed", "speed_troughs"]].sel(keypoint=KEYPOINT, individual=individual)
print(
    f"trial {trial} | keypoint {KEYPOINT!r} | individual {individual!r} | "
    f"{pinned.sizes['time']} frames @ {ds.attrs['fps']:.0f} fps"
)

trial 41 | keypoint 'beakTip' | individual np.str_('Crow1') | 1077 frames @ 200 fps


## Expand the mask

This dataset predates the variable-schema convention and still carries the legacy
`attrs["type"] = "changepoints"` spelling. Nothing migrates that
automatically on load — `schema.migrate_legacy_attrs` is a one-time,
explicit conversion you run yourself (a fresh file written with `describe`/
`add_changepoints_to_ds` never carried the legacy spelling, so it needs no
such call). The `project.yaml` route in the last cell below does this
migration for you automatically, once per session — only calling
`add_changepoint_features` directly, as here, needs the explicit step.

In [ ]:
pinned = schema.migrate_legacy_attrs(pinned)
print("speed_troughs attrs:", dict(pinned["speed_troughs"].attrs))

expanded = add_changepoint_features(pinned, sigmas=SIGMAS)
new_cols = [name for name in expanded.data_vars if name not in pinned.data_vars]
print(f"\n{len(new_cols)} new columns:")
for name in new_cols:
    print(" ", name)

## Plot every representation

Zoomed into a few seconds for a clearer picture than the whole (busy) trial.

In [ ]:
WINDOW_S = (1.5, 3.5)

time_full = expanded["time"].values
in_window = (time_full >= WINDOW_S[0]) & (time_full <= WINDOW_S[1])
expanded_win = expanded.isel(time=in_window)

time = expanded_win["time"].values
speed = expanded_win["speed"].values
mask = expanded_win["speed_troughs"].values
trough_times = time[mask.astype(bool)]

sigma_cols = [f"speed_troughs_cp_prox{i}" for i in range(len(SIGMAS))]  # named by rank, narrowest first

fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)

ax = axes[0]
ax.plot(time, speed, color="black", lw=1)
for t0 in trough_times:
    ax.axvline(t0, color="crimson", lw=0.6, alpha=0.5)
ax.set_ylabel("speed")
ax.set_title(f"raw speed (target feature) + {int(mask.sum())} troughs in this window")

ax = axes[1]
ax.step(time, mask, where="mid", color="crimson")
ax.set_ylabel("cp_binary")
ax.set_title("binary mask — the input to more_changepoint_features")

ax = axes[2]
for name, sigma in zip(sigma_cols, SIGMAS):
    ax.plot(time, expanded_win[name].values, label=f"sigma {sigma:g}")
ax.set_ylabel("proximity")
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Laplacian proximity — one curve per sigma, wider sigma = broader peak")

ax = axes[3]
for name, label in (("speed_troughs_cp_since", "since previous"), ("speed_troughs_cp_until", "until next")):
    ax.plot(time, expanded_win[name].values, label=label)
ax.plot(time, expanded_win["speed_troughs_cp_length"].values, color="gray", ls="--", label="segment length (log)")
ax.set_ylabel("offset / length")
ax.set_xlabel("time (s)")
ax.legend(loc="upper right", fontsize=8)
ax.set_title(f"offsets — samples since / until the nearest trough, clipped at {4 * max(SIGMAS):g} samples (default horizon = 4 × max sigma)")

plt.tight_layout()
plt.show()

## Turning it on for real

**Directly on your `TrialTree`** — same pattern as any other feature-engineering step,
just done once and saved:

```python
dt = dt.map_trials(lambda ds: add_changepoint_features(ds, sigmas=[2.0, 3.0, 5.0]))
dt.save(str(nc_path))
```

**In `project.yaml`**, so it runs once per session at `materialise`/`infer` time instead
of being baked into the file — name the raw masks under `inputs` (dims pinned the same
way as `features.columns`) and which column groups you want under `transforms`; the
generated columns are merged into `features.columns` for you, so you never spell out a
name like `speed_troughs_cp_since` by hand:

```yaml
features:
  changepoint_features:
    transforms: [proximity, offset, length]
    scale_by: speed
    inputs:
      speed_troughs: {keypoint: [beakTip, stickTip]}
```

`transforms` is a subset of `binary`, `proximity`, `offset`, `length` — `binary` is left out above
because it just duplicates the raw mask (only marked `normalise=0`), and this config
already doesn't select `speed_troughs` directly. `scale_by` names a feature whose values
scale the proximity columns by `exp(−x / mean x)`, so troughs at rest count for more than
a dip inside a movement; leave it out for the plain kernels. `sigmas`, `horizon` and `max_length` are left out on purpose:
materialise reads them off the curated label durations (half the 5th percentile for the horizon, the
95th percentile for `max_length`, the ladder horizon / (16, 8, 4) for the sigmas), records them in the
dataset's `columns.yaml` with a note saying so, and every later stage reads them back. Spell any of
them, in samples, to pin it. Every generated column already
carries `attrs["normalise"] = 0`, so none of them need a `preprocess.zscore_exclude` entry
either.